In [ ]:
import json

def load_json(filepath: str):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

races_data = load_json("assets/usable/races.json")

def trace_types(data, path=""):
    current_type = type(data).__name__

    new_path = f"{path}->{current_type}" if path else current_type

    if isinstance(data, list):
        for item in data:
            yield from trace_types(item, new_path)
    elif isinstance(data, dict):
        for value in data.values():
            yield from trace_types(value, new_path)
    else:
        yield new_path

paths = []
for type_path in trace_types(races_data):
    paths.append(type_path)

paths = list(set(paths))

for p in paths:
    print(p)

list->dict->list->dict->dict->list->str
list->dict->dict->str
list->dict->list->dict->list->dict->list->str
list->dict->bool
list->dict->list->dict->list->str
list->dict->list->dict->bool
list->dict->list->dict->list->dict->str
list->dict->list->dict->list->dict->list->list->str
list->dict->list->dict->str
list->dict->str
list->dict->list->dict->dict->int
list->dict->int
list->dict->list->str
list->dict->dict->int
list->dict->list->dict->dict->dict->dict->list->str
list->dict->list->dict->int


```plaintext
list->dict->list->dict->dict->list->str
list->dict->dict->str
list->dict->list->dict->list->dict->list->str
list->dict->bool
list->dict->list->dict->list->str
list->dict->list->dict->bool
list->dict->list->dict->list->dict->str
list->dict->list->dict->list->dict->list->list->str
list->dict->list->dict->str
list->dict->str
list->dict->list->dict->dict->int
list->dict->int
list->dict->list->str
list->dict->dict->int
list->dict->list->dict->dict->dict->dict->list->str
list->dict->list->dict->int
```

In [ ]:
import json
from math import floor
from pydantic import computed_field, BaseModel, Field, model_validator
from typing import Literal, Optional, Self


class Abilities(BaseModel):
    choose_amount: Optional[int]
    choose_items: Optional[
        list[Literal["strength", "dexterity", "constitution", "intelligence", "wisdom", "charisma"]]
    ]
    strength: int = 0
    dexterity: int = 0
    constitution: int = 0
    intelligence: int = 0
    wisdom: int = 0
    charisma: int = 0

    @computed_field
    @property
    def strength_mod(self) -> int:
        return floor(self.strength - 10 / 2)

    @computed_field
    @property
    def dexterity_mod(self) -> int:
        return floor(self.dexterity - 10 / 2)

    @computed_field
    @property
    def constitution_mod(self) -> int:
        return floor(self.constitution - 10 / 2)

    @computed_field
    @property
    def intelligence_mod(self) -> int:
        return floor(self.intelligence - 10 / 2)

    @computed_field
    @property
    def wisdom_mod(self) -> int:
        return floor(self.wisdom - 10 / 2)

    @computed_field
    @property
    def charisma_mod(self) -> int:
        return floor(self.charisma - 10 / 2)
    
    @model_validator(mode="after")
    def choose_abilities(self) -> Self:
        if self.choose_amount and self.choose_items:
            if len(self.choose_items) != self.choose_amount:
                raise ValueError(f"Chose {len(self.choose_items)}, can only choose {self.choose_amount} items.")
            for item in self.choose_items:
                pass
                # setattr(self, item, value)
        return self


class DiceMod(BaseModel):
    input_str: str
    amount: int = Field(ge=1, default=1)
    sides: int = 6

    @model_validator(mode="after")
    def assign_amount_sides(self) -> Self:
        amount, sides = self.input_str.split("d")
        self.sides = int(sides)
        self.amount = int(amount)
        return self


class HeightWeight(BaseModel):
    base_height: Optional[int]
    height_mod: Optional[DiceMod]
    base_weight: Optional[int]
    weight_mod: Optional[DiceMod]


class Age(BaseModel):
    current: int
    mature: int
    maximum: int
    is_mature: bool = False

    @model_validator(mode="after")
    def ensure_age_limits(self) -> Self:
        if self.current > self.maximum:
            self.current = self.maximum
        if self.current >= self.mature:
            self.is_mature = True
        return self


class ChooseResistance(BaseModel):
    choice: Literal[
        "fire", "poison", "cold", "psychic", "magic", "acid", "lightning", "necrotic"
    ]
    fire: bool = False
    poison: bool = False
    cold: bool = False
    psychic: bool = False
    magic: bool = False
    acid: bool = False
    lightning: bool = False
    necrotic: bool = False

    @model_validator(mode="after")
    def set_resistance_choice(self) -> Self:
        setattr(self, self.choice, True)
        return self


class Size(BaseModel):
    chosen_size: Literal["S", "M", "L"]
    small: bool = False
    medium: bool = False
    large: bool = False

    @model_validator(mode="after")
    def assign_size(self) -> Self:
        size_dict: dict[str, str] = {
            "S": "small",
            "M": "medium",
            "L": "large",
        }
        setattr(self, size_dict[self.chosen_size], True)
        return self


class Race(BaseModel):
    name: Literal["Dragonborn", "Dwarf", "Elf", "Gnome", "Half-Elf", "Half-Orc", "Halfling", "Human", "Tiefling"]
    size: Size
    speed: int = Field(ge=10, le=50)
    abilities: Abilities
    height_and_weight: HeightWeight
    age: Age
    language_proficiencies: list[str] = ["Common"]
    choose_resistance: ChooseResistance
    darkvision: Optional[int]
    tool_proficiencies: Optional[
        list[Literal["Smith's Tools", "Brewer's Supplies", "Mason's Tools"]]
    ]
    weapon_proficiencies: Optional[
        list[Literal["Battleaxe", "Handaxe", "Light Hammer", "Warhammer"]]
    ]
    skill_proficiencies: Optional[
        list[
            Literal[
                "Acrobatics",
                "Animal Handling",
                "Arcana",
                "Athletics",
                "Deception",
                "History",
                "Insight",
                "Intimidation",
                "Investigation",
                "Medicine",
                "Nature",
                "Perception",
                "Performance",
                "Persuasion",
                "Religion",
                "Sleight of Hand",
                "Stealth",
                "Survival",
            ]
        ]
    ]

    @model_validator(mode="after")
    def assign_model_information(self) -> Self:
        with open("assets/usable/races.json", "r", encoding="utf-8") as f:
            races_data = json.load(f)
        for race in races_data:
            if self.name == race["name"]:
                if "size" in race:
                    self.size = Size(chosen_size=race["size"][0])
                if "speed" in race:
                    self.speed = int(race["speed"])
                if "heightAndWeight" in race:
                    hw = race["heightAndWeight"]
                    base_height = None
                    height_mod = None
                    base_weight = None
                    weight_mod = None
                    if "baseHeight" in hw:
                        base_height = int(hw["baseHeight"])
                    if "heightMod" in hw:
                        height_mod = DiceMod(input_str=hw["heightMod"])
                    if "baseWeight" in hw:
                        base_weight = int(hw["baseWeight"])
                    if "weightMod" in hw:
                        weight_mod = DiceMod(input_str=hw["weightMod"])
                    

In [5]:
import json

def load_json(filepath: str):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

races_data = load_json("assets/usable/races.json")

for race in races_data:
    if "name" in race:
        print("name:", race["name"])
    if "size" in race:
        print("size:", race["size"])
    if "speed" in race:
        print("speed:", race["speed"])
    if "ability" in race:
        print("ability:", race["ability"])
    if "heightAndWeight" in race:
        print("heightAndWeight:", race["heightAndWeight"])
    if "age" in race:
        print("age:", race["age"])
    if "languageProficiencies" in race:
        print("languageProficiencies:", race["languageProficiencies"])
    if "resist" in race:
        print("resist:", race["resist"])
    if "darkvision" in race:
        print("darkvision:", race["darkvision"])
    if "toolProficiencies" in race:
        print("toolProficiencies:", race["toolProficiencies"])
    if "weaponProficiencies" in race:
        print("weaponProficiencies:", race["weaponProficiencies"])
    if "skillProficiencies" in race:
        print("skillProficiencies:", race["skillProficiencies"])
    print()

name: Dragonborn
size: ['M']
speed: 30
ability: [{'str': 2, 'cha': 1}]
heightAndWeight: {'baseHeight': 66, 'heightMod': '2d8', 'baseWeight': 175, 'weightMod': '2d6'}
age: {'mature': 15, 'max': 80}
languageProficiencies: [{'common': True, 'draconic': True}]
resist: [{'choose': {'from': ['acid', 'cold', 'fire', 'lightning', 'poison']}}]

name: Dwarf
size: ['M']
speed: 25
ability: [{'con': 2}]
age: {'mature': 20, 'max': 350}
languageProficiencies: [{'common': True, 'dwarvish': True}]
resist: ['poison']
darkvision: 60
toolProficiencies: [{'choose': {'from': ["smith's tools", "brewer's supplies", "mason's tools"]}}]
weaponProficiencies: [{'battleaxe|phb': True, 'handaxe|phb': True, 'light hammer|phb': True, 'warhammer|phb': True}]

name: Elf
size: ['M']
speed: 30
ability: [{'dex': 2}]
age: {'mature': 100, 'max': 750}
languageProficiencies: [{'common': True, 'elvish': True}]
darkvision: 60
skillProficiencies: [{'perception': True}]

name: Gnome
size: ['S']
speed: 25
ability: [{'int': 2}]
hei

```
name: Dragonborn
size: ['M']
speed: 30
ability: [{'str': 2, 'cha': 1}]
heightAndWeight: {'baseHeight': 66, 'heightMod': '2d8', 'baseWeight': 175, 'weightMod': '2d6'}
age: {'mature': 15, 'max': 80}
languageProficiencies: [{'common': True, 'draconic': True}]
resist: [{'choose': {'from': ['acid', 'cold', 'fire', 'lightning', 'poison']}}]

name: Dwarf
size: ['M']
speed: 25
ability: [{'con': 2}]
age: {'mature': 20, 'max': 350}
languageProficiencies: [{'common': True, 'dwarvish': True}]
resist: ['poison']
darkvision: 60
toolProficiencies: [{'choose': {'from': ["smith's tools", "brewer's supplies", "mason's tools"]}}]
weaponProficiencies: [{'battleaxe|phb': True, 'handaxe|phb': True, 'light hammer|phb': True, 'warhammer|phb': True}]

name: Elf
size: ['M']
speed: 30
ability: [{'dex': 2}]
age: {'mature': 100, 'max': 750}
languageProficiencies: [{'common': True, 'elvish': True}]
darkvision: 60
skillProficiencies: [{'perception': True}]

name: Gnome
size: ['S']
speed: 25
ability: [{'int': 2}]
heightAndWeight: {'baseHeight': 35, 'heightMod': '2d4', 'baseWeight': 35}
age: {'mature': 40, 'max': 500}
languageProficiencies: [{'common': True, 'gnomish': True}]
darkvision: 60

name: Half-Elf
size: ['M']
speed: 30
ability: [{'cha': 2, 'choose': {'from': ['str', 'dex', 'con', 'int', 'wis'], 'count': 2}}]
heightAndWeight: {'baseHeight': 57, 'heightMod': '2d8', 'baseWeight': 110, 'weightMod': '2d4'}
age: {'mature': 20, 'max': 180}
languageProficiencies: [{'common': True, 'elvish': True, 'anyStandard': 1}]
darkvision: 60
skillProficiencies: [{'any': 2}]

name: Half-Orc
size: ['M']
speed: 30
ability: [{'str': 2, 'con': 1}]
heightAndWeight: {'baseHeight': 58, 'heightMod': '2d10', 'baseWeight': 140, 'weightMod': '2d6'}
age: {'mature': 14, 'max': 75}
languageProficiencies: [{'common': True, 'orc': True}]
darkvision: 60
skillProficiencies: [{'intimidation': True}]

name: Halfling
size: ['S']
speed: 25
ability: [{'dex': 2}]
heightAndWeight: {'baseHeight': 31, 'heightMod': '2d4', 'baseWeight': 35}
age: {'mature': 20, 'max': 250}
languageProficiencies: [{'common': True, 'halfling': True}]

name: Human
size: ['M']
speed: 30
heightAndWeight: {'baseHeight': 56, 'heightMod': '2d10', 'baseWeight': 110, 'weightMod': '2d4'}
age: {'mature': 20, 'max': 100}
languageProficiencies: [{'common': True, 'anyStandard': 1}]

name: Tiefling
size: ['M']
speed: 30
ability: [{'cha': 2, 'int': 1}]
heightAndWeight: {'baseHeight': 57, 'heightMod': '2d8', 'baseWeight': 110, 'weightMod': '2d4'}
age: {'mature': 20, 'max': 100}
languageProficiencies: [{'common': True, 'infernal': True}]
resist: ['fire']
darkvision: 60
```